# 文本分类的神经网络模型

所展示的模型和数据均来源于互联网[中文文本分类]()，提供了基于PyTorch实现的TextCNN，TextRNN，FastText，TextRCNN，BiLSTM_Attention, DPCNN, Transformer等。原作者的模型介绍、数据流动过程参考[这里](https://zhuanlan.zhihu.com/p/73176084)  

经过进一步整理以便于理解和使用，除了数据文件之外，本讲稿内的代码是自包含的。

数据以字为单位输入模型，预训练词向量使用 [搜狗新闻 Word+Character 300d](https://github.com/Embedding/Chinese-Word-Vectors)，[点这里下载](https://pan.baidu.com/s/14k-9jsspp43ZhMxqPmsWMQ)  ，腾讯词向量也容易在网上下载到。

In [1]:
import os
import time
import pickle as pkl
from tqdm import tqdm
from sklearn import metrics
import numpy as np

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
np.random.seed(1)
torch.manual_seed(1)
torch.cuda.manual_seed_all(1)

In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# device = 'cpu'
device

device(type='cuda', index=0)

## 数据集

从[THUCNews](http://thuctc.thunlp.org/)中抽取了20万条新闻标题

- 文本长度在20到30之间
- 10个类别（财经、房产、股票、教育、科技、社会、时政、体育、游戏、娱乐）,每类2万条。
- 训练集(train.txt)18万，验证集(val.txt)1万条，测试集(test.txt)1万

In [5]:
n_vocab = 0   # 运行时复制

嵌入预训练模型与词表

In [6]:
embedding_pretrained = torch.tensor(np.load('embedding_Tencent.npz')["embeddings"].astype('float32'))

In [7]:
embed = embedding_pretrained.size(1)  if embedding_pretrained is not None else 300      # 字向量维度
print(embed)

200


数据集定义

In [8]:
class_list = [x.strip() for x in open('THUCNews/class.txt', encoding='utf-8').readlines()]  
num_classes = len(class_list)
print(num_classes)

10


In [9]:
MAX_VOCAB_SIZE = 10000  # 词表长度限制
UNK, PAD = '<UNK>', '<PAD>'  # 未知字，padding符号

In [10]:
# 辅助函数，用于根据数据集实时建立词表
def build_vocab(file_path, tokenizer, max_size, min_freq):
    vocab_dic = {}
    with open(file_path, 'r', encoding='UTF-8') as f:
        for line in tqdm(f):
            lin = line.strip()
            if not lin:
                continue
            content = lin.split('\t')[0]
            for word in tokenizer(content):
                vocab_dic[word] = vocab_dic.get(word, 0) + 1
        vocab_list = sorted([_ for _ in vocab_dic.items() if _[1] >= min_freq], key=lambda x: x[1], reverse=True)[:max_size]
        vocab_dic = {word_count[0]: idx for idx, word_count in enumerate(vocab_list)}
        vocab_dic.update({UNK: len(vocab_dic), PAD: len(vocab_dic) + 1})
    return vocab_dic

In [11]:
# 建立词表
use_word = True
vocab_path = 'THUCNews/vocab.pkl'
if use_word:
    tokenizer = lambda x: x.split(' ')  # 以空格隔开，word-level
else:
    tokenizer = lambda x: [y for y in x]  # char-level
if os.path.exists(vocab_path):
    vocab = pkl.load(open(vocab_path, 'rb'))
else:
    vocab = build_vocab(txt_path, tokenizer=tokenizer, max_size=MAX_VOCAB_SIZE, min_freq=1)
    pkl.dump(vocab, open(vocab_path, 'wb'))
print(f"Vocab size: {len(vocab)}")

Vocab size: 4762


In [12]:
def load_dataset(path, pad_size=32):
    contents = []
    with open(path, 'r', encoding='UTF-8') as f:
        for line in tqdm(f):
            lin = line.strip()
            if not lin:
                continue
            content, label = lin.split('\t')
            words_line = []
            token = tokenizer(content)
            seq_len = len(token)
            if pad_size:
                if len(token) < pad_size:
                    token.extend([PAD] * (pad_size - len(token)))
                else:
                    token = token[:pad_size]
                    seq_len = pad_size
            # word to id
            for word in token:
                words_line.append(vocab.get(word, vocab.get(UNK)))
            contents.append((words_line, int(label), seq_len))
    return contents  # [([...], 0), ([...], 1), ...]

In [13]:
pad_size = 32
train_data = load_dataset('THUCNews/train.txt', pad_size)
val_data = load_dataset('THUCNews/val.txt', pad_size)
test_data = load_dataset('THUCNews/test.txt', pad_size)

180000it [00:02, 78998.08it/s]
10000it [00:00, 107480.94it/s]
10000it [00:00, 107042.61it/s]


In [14]:
class DatasetIterater(object):
    def __init__(self, batches, batch_size, device):
        self.batch_size = batch_size
        self.batches = batches
        self.n_batches = len(batches) // batch_size
        self.residue = False  # 记录batch数量是否为整数
        if len(batches) % self.n_batches != 0:
            self.residue = True
        self.index = 0

    def _to_tensor(self, datas):
        x = torch.LongTensor([_[0] for _ in datas]).to(device)
        y = torch.LongTensor([_[1] for _ in datas]).to(device)

        # pad前的长度(超过pad_size的设为pad_size)
        seq_len = torch.LongTensor([_[2] for _ in datas]).to(device)
        return (x, seq_len), y

    def __next__(self):
        if self.residue and self.index == self.n_batches:
            batches = self.batches[self.index * self.batch_size: len(self.batches)]
            self.index += 1
            batches = self._to_tensor(batches)
            return batches

        elif self.index >= self.n_batches:
            self.index = 0
            raise StopIteration
        else:
            batches = self.batches[self.index * self.batch_size: (self.index + 1) * self.batch_size]
            self.index += 1
            batches = self._to_tensor(batches)
            return batches

    def __iter__(self):
        return self

    def __len__(self):
        if self.residue:
            return self.n_batches + 1
        else:
            return self.n_batches

In [45]:
batch_size = 100
train_iter = DatasetIterater(train_data, batch_size, device)
val_iter = DatasetIterater(val_data, batch_size, device)
test_iter = DatasetIterater(test_data, batch_size, device)

In [18]:
train_iter.__len__()

1800

### 2.1. RNN(LSTM)

In [20]:
class TextRNN(nn.Module):
    def __init__(self, num_classes, embed, n_vocab, hidden_size, num_layers, dropout):
        super(TextRNN, self).__init__()

        self.rnn = nn.LSTM(embed, hidden_size, num_layers=num_layers, bidirectional=True, batch_first=True, dropout=dropout)
        if embedding_pretrained is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_pretrained, freeze=False)
        else:
            self.embedding = nn.Embedding(n_vocab, embed, padding_idx=n_vocab - 1)

        self.fc = nn.Linear(hidden_size * num_layers, num_classes)

        self.dropout = nn.Dropout(dropout)
        self.batch_first = True

    def forward(self, text):
        text, text_lengths = text
        # 按照句子长度从大到小排序

        sorted_seq_lengths, indices = torch.sort(text_lengths, descending=True)
        _, desorted_indices = torch.sort(indices, descending=False)
        text = text[indices]

        # text = [batch size,sent len]
        embedded = self.dropout(self.embedding(text)).float()
        # embedded = [batch size, sent len, emb dim]

        # pack sequence
        sorted_seq_lengths = sorted_seq_lengths.to("cpu")
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, sorted_seq_lengths, batch_first=self.batch_first)
        self.rnn.flatten_parameters()

        # output (seq_len, batch, num_directions * hidden_size)
        # hidden (num_layers * num_directions, batch, hidden_size)
        packed_output, (hidden, cell) = self.rnn(packed_embedded)

        # unpack sequence
        output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=self.batch_first)
        # 把句子序列再调整成输入时的顺序
        output = output[desorted_indices]
        # output = [batch_size,seq_len,hidden_dim * num_directionns ]
        batch_size, max_seq_len, hidden_dim = output.shape
        hidden = torch.mean(torch.reshape(hidden, [batch_size, -1, hidden_dim]), dim=1)
        output = torch.mean(output, dim=1)
        fc_input = self.dropout(output + hidden)
        out = self.fc(fc_input)

        return out

In [21]:
# RNN模型相关参数
dropout = 0.5                                              
hidden_size = 128
num_layers = 2

In [ ]:
model = TextRNN(num_classes, embed, n_vocab, hidden_size, num_layers, dropout)

### 2.2. DPCNN

In [30]:
class TextDPCNN(nn.Module):
    '''Deep Pyramid Convolutional Neural Networks for Text Categorization'''
    def __init__(self, num_classes, embed, n_vocab, num_filters=250):
        super(TextDPCNN, self).__init__()
        if embedding_pretrained is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_pretrained, freeze=False)
        else:
            self.embedding = nn.Embedding(n_vocab, embed, padding_idx=n_vocab - 1)
        self.conv_region = nn.Conv2d(1, num_filters, (3, embed), stride=1)
        self.conv = nn.Conv2d(num_filters, num_filters, (3, 1), stride=1)
        self.max_pool = nn.MaxPool2d(kernel_size=(3, 1), stride=2)
        self.padding1 = nn.ZeroPad2d((0, 0, 1, 1))  # top bottom
        self.padding2 = nn.ZeroPad2d((0, 0, 0, 1))  # bottom
        self.relu = nn.ReLU()
        self.fc = nn.Linear(num_filters, num_classes)

    def forward(self, x):
        x = x[0]
        x = self.embedding(x)
        x = x.unsqueeze(1)  # [batch_size, 250, seq_len, 1]
        x = self.conv_region(x)  # [batch_size, 250, seq_len-3+1, 1]

        x = self.padding1(x)  # [batch_size, 250, seq_len, 1]
        x = self.relu(x)
        x = self.conv(x)  # [batch_size, 250, seq_len-3+1, 1]
        x = self.padding1(x)  # [batch_size, 250, seq_len, 1]
        x = self.relu(x)
        x = self.conv(x)  # [batch_size, 250, seq_len-3+1, 1]
        while x.size()[2] > 2:
            x = self._block(x)
        x = x.squeeze()  # [batch_size, num_filters(250)]
        x = self.fc(x)
        return x

    def _block(self, x):
        x = self.padding2(x)
        px = self.max_pool(x)

        x = self.padding1(px)
        x = F.relu(x)
        x = self.conv(x)

        x = self.padding1(x)
        x = F.relu(x)
        x = self.conv(x)

        # Short Cut
        x = x + px
        return x


In [31]:
num_filters = 250
model = TextDPCNN(num_classes, embed, n_vocab, num_filters)

### 2.3. Transformer

下面是一些transformer所需具备的最简洁的代码片段，是通用的，从原作者的代码中拷贝过来未经任何改动。按照常规可以单独放在一个.py文件中

In [ ]:
class Encoder(nn.Module):
    def __init__(self, dim_model, num_head, hidden, dropout):
        super(Encoder, self).__init__()
        self.attention = Multi_Head_Attention(dim_model, num_head, dropout)
        self.feed_forward = Position_wise_Feed_Forward(dim_model, hidden, dropout)

    def forward(self, x):
        out = self.attention(x)
        out = self.feed_forward(out)
        return out

![MHA](imgs/attention-to-multihead.png)

In [49]:
class Multi_Head_Attention(nn.Module):
    def __init__(self, dim_model, num_head, dropout=0.0):
        super(Multi_Head_Attention, self).__init__()
        self.num_head = num_head
        assert dim_model % num_head == 0
        self.dim_head = dim_model // self.num_head
        self.fc_Q = nn.Linear(dim_model, num_head * self.dim_head)
        self.fc_K = nn.Linear(dim_model, num_head * self.dim_head)
        self.fc_V = nn.Linear(dim_model, num_head * self.dim_head)
 #       self.attention = Scaled_Dot_Product_Attention()
        self.fc = nn.Linear(num_head * self.dim_head, dim_model)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(dim_model)

    def forward(self, x):
        batch_size = x.size(0)
        Q = self.fc_Q(x)
        K = self.fc_K(x)
        V = self.fc_V(x)
        Q = Q.view(batch_size * self.num_head, -1, self.dim_head)
        K = K.view(batch_size * self.num_head, -1, self.dim_head)
        V = V.view(batch_size * self.num_head, -1, self.dim_head)
        # if mask:  # TODO
        #     mask = mask.repeat(self.num_head, 1, 1)  # TODO change this
        
        attention = torch.matmul(Q, K.permute(0, 2, 1))
        scale = K.size(-1) ** -0.5  # 缩放因子
  #      context = self.attention(Q, K, V, scale)
        if scale:
            attention = attention * scale
        # if mask:  # TODO change this
        #     attention = attention.masked_fill_(mask == 0, -1e9)
        attention = F.softmax(attention, dim=-1)
        context = torch.matmul(attention, V)

        context = context.view(batch_size, -1, self.dim_head * self.num_head)
        out = self.fc(context)
        out = self.dropout(out)
        out = out + x  # 残差连接
        out = self.layer_norm(out)
        return out

In [ ]:
class Position_wise_Feed_Forward(nn.Module):
    def __init__(self, dim_model, hidden, dropout=0.0):
        super(Position_wise_Feed_Forward, self).__init__()
        self.fc1 = nn.Linear(dim_model, hidden)
        self.fc2 = nn.Linear(hidden, dim_model)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(dim_model)

    def forward(self, x):
        out = self.fc1(x)
        out = F.relu(out)
        out = self.fc2(out)
        out = self.dropout(out)
        out = out + x  # 残差连接
        out = self.layer_norm(out)
        return out

In [50]:
import copy    # 用于Layer的深层拷贝

In [55]:
class Positional_Encoding(nn.Module):
    def __init__(self, embed, pad_size, dropout, device):
        super(Positional_Encoding, self).__init__()
        self.device = device
        self.pe = torch.tensor([[pos / (10000.0 ** (i // 2 * 2.0 / embed)) for i in range(embed)] for pos in range(pad_size)])
        self.pe[:, 0::2] = np.sin(self.pe[:, 0::2])
        self.pe[:, 1::2] = np.cos(self.pe[:, 1::2])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = x + nn.Parameter(self.pe, requires_grad=False).to(self.device)
        out = self.dropout(out)
        return out

![transformer](imgs/transformer-structure.png)

In [56]:
class TextTransformer(nn.Module):
    def __init__(self, num_classes, embed, n_vocab, num_encoder, dim_model, num_head, hidden, pad_size, dropout, device):
        super(TextTransformer, self).__init__()
        if embedding_pretrained is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_pretrained, freeze=False)
        else:
            self.embedding = nn.Embedding(n_vocab, embed, padding_idx=n_vocab - 1)

        self.postion_embedding = Positional_Encoding(embed, pad_size, dropout, device)
        self.encoder = Encoder(dim_model, num_head, hidden, dropout)
        self.encoders = nn.ModuleList([
            copy.deepcopy(self.encoder)
            # Encoder(config.dim_model, config.num_head, config.hidden, config.dropout)
            for _ in range(num_encoder)])

        self.fc1 = nn.Linear(pad_size * dim_model, num_classes)
        # self.fc2 = nn.Linear(config.last_hidden, config.num_classes)
        # self.fc1 = nn.Linear(config.dim_model, config.num_classes)

    def forward(self, x):
        out = self.embedding(x[0])
        out = self.postion_embedding(out)
        for encoder in self.encoders:
            out = encoder(out)
        out = out.view(out.size(0), -1)
        # out = torch.mean(out, 1)
        out = self.fc1(out)
        return out

In [57]:
dropout = 0.5
num_encoder = 2
num_head = 5
dim_model = 200
hidden = 1024

In [58]:
model = TextTransformer(num_classes, embed, n_vocab, num_encoder, dim_model, num_head, hidden, pad_size, dropout, device)

In [43]:
device

device(type='cuda', index=1)

### 2.4. FastText

In [38]:
class FastText(nn.Module):
    def __init__(self, num_classes, embed, n_vocab, n_gram_vocab, hidden_size, dropout):
        super(FastText, self).__init__()
        if embedding_pretrained is not None:
            self.embedding = nn.Embedding.from_pretrained(embedding_pretrained, freeze=False)
        else:
            self.embedding = nn.Embedding(n_vocab, embed, padding_idx=n_vocab - 1)
        self.embedding_ngram2 = nn.Embedding(n_gram_vocab, embed)
        self.embedding_ngram3 = nn.Embedding(n_gram_vocab, embed)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(embed * 3, hidden_size)
        # self.dropout2 = nn.Dropout(config.dropout)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        out_word = self.embedding(x[0])
        out_bigram = self.embedding_ngram2(x[2])
        out_trigram = self.embedding_ngram3(x[3])
        out = torch.cat((out_word, out_bigram, out_trigram), -1)

        out = out.mean(dim=1)
        out = self.dropout(out)
        out = self.fc1(out)
        out = F.relu(out)
        out = self.fc2(out)
        return out

In [40]:
n_gram_vocab = 250499
hidden_size = 256
dropout = 0.5

In [59]:
model = FastText(num_classes, embed, n_vocab, n_gram_vocab, hidden_size, dropout)

注意到FastText的forward中需要用到更多的embedding层，包括上述定义中的out_bigram和out_trigram，因此需要在dataIterator中加入相应的数据x[2],x[3]。这里我们重新定义DataSetIterator

In [66]:
class DatasetIterater(object):
    def __init__(self, batches, batch_size, device):
        self.batch_size = batch_size
        self.batches = batches
        self.n_batches = len(batches) // batch_size
        self.residue = False  # 记录batch数量是否为整数 
        if len(batches) % self.n_batches != 0:
            self.residue = True
        self.index = 0
        self.device = device

    def _to_tensor(self, datas):
        # xx = [xxx[2] for xxx in datas]
        # indexx = np.argsort(xx)[::-1]
        # datas = np.array(datas)[indexx]
        x = torch.LongTensor([_[0] for _ in datas]).to(self.device)
        y = torch.LongTensor([_[1] for _ in datas]).to(self.device)
        bigram = torch.LongTensor([_[3] for _ in datas]).to(self.device)
        trigram = torch.LongTensor([_[4] for _ in datas]).to(self.device)

        # pad前的长度(超过pad_size的设为pad_size)
        seq_len = torch.LongTensor([_[2] for _ in datas]).to(self.device)
        return (x, seq_len, bigram, trigram), y

    def __next__(self):
        if self.residue and self.index == self.n_batches:
            batches = self.batches[self.index * self.batch_size: len(self.batches)]
            self.index += 1
            batches = self._to_tensor(batches)
            return batches

        elif self.index >= self.n_batches:
            self.index = 0
            raise StopIteration
        else:
            batches = self.batches[self.index * self.batch_size: (self.index + 1) * self.batch_size]
            self.index += 1
            batches = self._to_tensor(batches)
            return batches

    def __iter__(self):
        return self

    def __len__(self):
        if self.residue:
            return self.n_batches + 1
        else:
            return self.n_batches

In [56]:
def biGramHash(sequence, t, buckets):
    t1 = sequence[t - 1] if t - 1 >= 0 else 0
    return (t1 * 14918087) % buckets

def triGramHash(sequence, t, buckets):
    t1 = sequence[t - 1] if t - 1 >= 0 else 0
    t2 = sequence[t - 2] if t - 2 >= 0 else 0
    return (t2 * 14918087 * 18408749 + t1 * 14918087) % buckets

In [57]:
def load_dataset(path, pad_size, n_gram_vocab):
    contents = []
    with open(path, 'r', encoding='UTF-8') as f:
        for line in tqdm(f):
            lin = line.strip()
            if not lin:
                continue
            content, label = lin.split('\t')
            words_line = []
            token = tokenizer(content)
            seq_len = len(token)
            if pad_size:
                if len(token) < pad_size:
                    token.extend([PAD] * (pad_size - len(token)))
                else:
                    token = token[:pad_size]
                    seq_len = pad_size
            # word to id
            for word in token:
                words_line.append(vocab.get(word, vocab.get(UNK)))
            
            # fasttext ngram
            buckets = n_gram_vocab
            bigram = []
            trigram = []
            # ------ngram------
            for i in range(pad_size):
                bigram.append(biGramHash(words_line, i, buckets))
                trigram.append(triGramHash(words_line, i, buckets))
            # -----------------                
            contents.append((words_line, int(label), seq_len, bigram, trigram))
    return contents  # [([...], 0), ([...], 1), ...]

In [58]:
pad_size = 32
train_data = load_dataset('THUCNews/train.txt', pad_size, n_gram_vocab)
val_data = load_dataset('THUCNews/val.txt', pad_size, n_gram_vocab)
test_data = load_dataset('THUCNews/test.txt', pad_size, n_gram_vocab)

180000it [00:08, 22396.08it/s]
10000it [00:00, 26021.92it/s]
10000it [00:00, 26066.72it/s]


In [67]:
batch_size = 256
train_iter = DatasetIterater(train_data, batch_size, device)
val_iter = DatasetIterater(val_data, batch_size, device)
test_iter = DatasetIterater(test_data, batch_size, device)

## 3. 开始训练 - 通用代码

In [27]:
from tensorboardX import SummaryWriter

In [59]:
# 具体model的声明，在第2节中实现
model = model.to(device)
model.eval()

TextTransformer(
  (embedding): Embedding(4762, 200)
  (postion_embedding): Positional_Encoding(
    (dropout): Dropout(p=0.5, inplace=False)
  )
  (encoder): Encoder(
    (attention): Multi_Head_Attention(
      (fc_Q): Linear(in_features=200, out_features=200, bias=True)
      (fc_K): Linear(in_features=200, out_features=200, bias=True)
      (fc_V): Linear(in_features=200, out_features=200, bias=True)
      (fc): Linear(in_features=200, out_features=200, bias=True)
      (dropout): Dropout(p=0.5, inplace=False)
      (layer_norm): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
    )
    (feed_forward): Position_wise_Feed_Forward(
      (fc1): Linear(in_features=200, out_features=1024, bias=True)
      (fc2): Linear(in_features=1024, out_features=200, bias=True)
      (dropout): Dropout(p=0.5, inplace=False)
      (layer_norm): LayerNorm((200,), eps=1e-05, elementwise_affine=True)
    )
  )
  (encoders): ModuleList(
    (0): Encoder(
      (attention): Multi_Head_Attention(
  

In [29]:
# 权重初始化，默认xavier
def init_network(model, method='xavier', exclude='embedding', seed=123):
    for name, w in model.named_parameters():
        if exclude not in name:
            if 'weight' in name:
                if method == 'xavier':
                    nn.init.xavier_normal_(w)
                elif method == 'kaiming':
                    nn.init.kaiming_normal_(w)
                else:
                    nn.init.normal_(w)
            elif 'bias' in name:
                nn.init.constant_(w, 0)
            else:
                pass

In [60]:
model.train()
# init_network(model) # do not work for transformer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# 学习率指数衰减，每次epoch：学习率 = gamma * 学习率
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
# writer = SummaryWriter(log_dir='./logs/' + time.strftime('%m-%d_%H.%M', time.localtime()))

In [61]:
num_epochs = 10

In [ ]:
start_time = time.time()
#flag = False  # 记录是否很久没有效果提升
total_batch = 0  # 记录进行到多少batch
#last_improve = 0  # 记录上次验证集loss下降的batch数
#dev_best_loss = float('inf')
for epoch in range(num_epochs):
    print('Epoch [{}/{}]'.format(epoch + 1, num_epochs))
    # scheduler.step() # 学习率衰减
    for i, (trains, labels) in enumerate(train_iter):
        
        # 1. 控制模型/状态方程/正向推理
        outputs = model(trains)        
        # 2. 计算 代价/目标/损失 函数
        loss = F.cross_entropy(outputs, labels)
        # 3. 敏感性分析/误差反向传播
        model.zero_grad()
        loss.backward()
        # 4. 优化器 迭代一步
        optimizer.step()
        
        if total_batch % 100 == 0:
            # 每多少轮输出在训练集和验证集上的效果
            true = labels.data.cpu()
            predic = torch.max(outputs.data, 1)[1].cpu()
            train_acc = metrics.accuracy_score(true, predic)
#            dev_acc, dev_loss = evaluate(config, model, dev_iter)
#            if dev_loss < dev_best_loss:
#                dev_best_loss = dev_loss
#                torch.save(model.state_dict(), config.save_path)
#                improve = '*'
#                last_improve = total_batch
#            else:
#                improve = ''
#            time_dif = get_time_dif(start_time)
            curr_time = time.time()
            time_dif = curr_time - start_time
            start_time = curr_time
            
            msg = 'Iter: {0:>6},  Train Loss: {1:>8.6},  Train Acc: {2:>6.2%}, Time: {3:>5.3}' #,  Val Loss: {3:>5.2},  Val Acc: {4:>6.2%},  Time: {5} {6}'
            print(msg.format(total_batch, loss.item(), train_acc, time_dif)) #, dev_loss, dev_acc, improve))
#            writer.add_scalar("loss/train", loss.item(), total_batch)
#            writer.add_scalar("loss/dev", dev_loss, total_batch)
#            writer.add_scalar("acc/train", train_acc, total_batch)
#            writer.add_scalar("acc/dev", dev_acc, total_batch)
            model.train()
    
        total_batch += 1
       # if total_batch - last_improve > config.require_improvement:
            # 验证集loss超过1000batch没下降，结束训练
       #     print("No optimization for a long time, auto-stopping...")
       #     flag = True
       #     break
#    if flag:
#        break
#writer.close()
#test(config, model, test_iter)

Epoch [1/10]
Iter:      0,  Train Loss:  2.37872,  Train Acc:  5.00%, Time: 0.611
Iter:    100,  Train Loss:   2.2962,  Train Acc: 17.00%, Time:  2.64
Iter:    200,  Train Loss:  2.18817,  Train Acc: 13.00%, Time:  2.69
Iter:    300,  Train Loss:  2.27308,  Train Acc: 21.00%, Time:  2.73
Iter:    400,  Train Loss:   2.2489,  Train Acc: 19.00%, Time:  2.73
Iter:    500,  Train Loss:  2.16075,  Train Acc: 15.00%, Time:  2.73
Iter:    600,  Train Loss:  2.35722,  Train Acc: 15.00%, Time:  2.73
Iter:    700,  Train Loss:  2.21971,  Train Acc: 13.00%, Time:  2.71
Iter:    800,  Train Loss:  2.21885,  Train Acc: 19.00%, Time:   2.7
Iter:    900,  Train Loss:  2.21527,  Train Acc: 15.00%, Time:  2.73
Iter:   1000,  Train Loss:  2.17228,  Train Acc: 12.00%, Time:  2.73
Iter:   1100,  Train Loss:  2.18081,  Train Acc: 22.00%, Time:  2.71
Iter:   1200,  Train Loss:  2.19798,  Train Acc: 18.00%, Time:  2.73
Iter:   1300,  Train Loss:  2.23252,  Train Acc: 15.00%, Time:  2.69
Iter:   1400,  Train 

5

## 4. 测试

In [48]:
# 解析一个列表的例子
texts = ['反映：我居住在xx市xx街道，近期我要办理暂住证，xx社区一位民警联系让我去社区办理暂住证，我对此有疑问，咨询部门办理暂住证应去何处，是否要收费，请告知具体流程',
         '反映：xx市xx街道xxx村附近的xx水泥制品厂，目前在制作石粉，风大的时候粉尘全部吹到村民屋里，之前环保部门已要求其整改，但现在还未进行环保公示，就在制作石粉，请部门核实整治。',
         '反映：xx市xx路xx幼儿园门口，每天放学时间段，有很多培训机构在门口发广告，让家长填写相关信息，希望部门核实对该处乱发广告一事加强监管。 （来电人不便留姓名）',
         '反映：20xx年1月至1x月，我在（身份证号：31xxxxxxxxxxxx19xx）xx市xx镇xxxx有限公司担任厨师，签订了劳动合同，现发现20xx年8月、9月、10月的社保显示未到账，希望部门核实，要求公司给我补齐这三个月的社保。',
]